# PEMS-SF 分组产物整理 (依赖 v3.5)
说明：已在 v3.5 生成 Stage5-figure/groups_g1_g5.json。此 Notebook 仅负责：
- 加载并对齐分组到 stations_list 顺序，生成布尔掩码。
- 轻量审计 train/test 的形状一致性。
- 导出统一索引与掩码供训练直接使用。
不在此 Notebook 重做聚类或‘最近中心’分配。

In [ ]:
import pathlib, json, re
import numpy as np
ROOT = pathlib.Path('.')
DATA_DIR = ROOT / 'data'
STAGE5_DIR = ROOT / 'Stage5-figure'
OUT_DIR = DATA_DIR / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('数据目录:', DATA_DIR.resolve())

In [ ]:
# 基础解析：每日矩阵与标签 (与 data_process 保持一致)
def parse_day_matrix(line: str, expected_rows=963, expected_cols=144):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if re.match(r'^-?\d+(\.\d+)?$', x)]
        if nums: data.append(nums)
    if not data: return None
    max_len = max(len(rr) for rr in data)
    arr = np.full((len(data), max_len), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit: break
            mat = parse_day_matrix(line)
            if mat is None: continue
            yield mat

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if re.match(r'^\d+$', x)], dtype=int)

In [ ]:
# 加载基础文件
train_path = DATA_DIR / 'PEMS_train'
test_path  = DATA_DIR / 'PEMS_test'
labels_train = load_labels(DATA_DIR / 'PEMS_trainlabels')
labels_test  = load_labels(DATA_DIR / 'PEMS_testlabels')
station_ids_txt = (DATA_DIR / 'stations_list').read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
station_ids = [int(x) for x in station_ids_txt.split() if re.match(r'^\d+$', x)]
assert len(station_ids) == 963, f'stations_list 数量异常: {len(station_ids)}'
print('Train days:', len(labels_train), 'Test days:', len(labels_test), 'Stations:', len(station_ids))

In [ ]:
# 加载 v3.5 分组并生成掩码（对齐 stations_list 顺序）
groups_index_path = STAGE5_DIR / 'groups_g1_g5.json'
if not groups_index_path.exists():
    raise FileNotFoundError('缺少 Stage5-figure/groups_g1_g5.json，请先运行 v3.5')
groups_index = json.loads(groups_index_path.read_text(encoding='utf-8'))
group_masks = {g: np.array([sid in set(groups_index.get(g, [])) for sid in station_ids], dtype=bool) for g in ['g1','g2','g3','g4','g5']}
print('对齐掩码：', {g: int(m.sum()) for g,m in group_masks.items()})

In [ ]:
# 轻量审计：前30天形状是否为 963×144
def audit_split(path: pathlib.Path, expect_shape=(963,144), limit=30):
    ok, bad = 0, []
    for i, mat in enumerate(iter_day_matrices(path, limit=limit)):
        if mat.shape == expect_shape: ok += 1
        else: bad.append((i, mat.shape))
    return {'iter': i+1 if 'i' in locals() else 0, 'ok': ok, 'bad': bad}
print('Train audit:', audit_split(train_path))
print('Test  audit:', audit_split(test_path))

In [ ]:
# 导出统一索引与掩码（训练/测试均消费同一分组）
export = {
  'station_ids': station_ids,
  'groups_index': groups_index,
  'meta': { 'train_days': len(labels_train), 'test_days': len(labels_test), 'stations': len(station_ids) }
}
(OUT_DIR / 'pems_sf_groups_index.json').write_text(json.dumps(export, ensure_ascii=False, indent=2), encoding='utf-8')
np.savez(OUT_DIR / 'pems_sf_groups_masks.npz', **{f'{k}_mask': v.astype(np.bool_) for k, v in group_masks.items()}, station_ids=np.array(station_ids, dtype=np.int32))
print('✓ 导出: ', (OUT_DIR / 'pems_sf_groups_index.json').resolve())
print('✓ 导出: ', (OUT_DIR / 'pems_sf_groups_masks.npz').resolve())

## 训练端使用说明
- 读取 data/processed/pems_sf_groups_index.json 获取 station_ids 与分组索引。
- 或读取 data/processed/pems_sf_groups_masks.npz 获取布尔掩码，按 stations_list 顺序与每日矩阵对齐。
- 训练端可直接遍历每日矩阵，应用目标组掩码，构造样本与标签。